# Step 1 - Data preparation: load, audit, clean, build the hourly grid

**CSC3119 AI Deployment & Scalability - Assignment 4**

This is the first of six notebooks. Together they build one pipeline that reads the raw AirQo CSV and produces a saved model, an August forecast, a drift report and a deployment package. Each notebook is self-contained and prints the reason behind every decision, so the pipeline can be explained line by line.

### What this notebook does

| Section | Purpose |
|---|---|
| 1 | Load the CSV and audit it: shape, types, nulls, raw zeros, duplicates |
| 2 | Establish **when** readings exist (the file is not what its name suggests) |
| 3 | Apply the brief's cleaning rules, with a justification for each |
| 4 | Reindex every sensor onto every hour and interpolate short gaps |
| 5 | Save the prepared tables, the data dictionary and a QC ledger |
| 6 | Draw coverage figures |

### Why preparation deserves its own notebook

The brief asks for *statistically justified* treatment of missing data, not blind deletion. Roughly a third of weather readings are missing or are the impossible value zero, and half the `site_id` column is empty. How we handle those decides whether the downstream model is describing Kampala's air or describing the behaviour of its modems.

Guiding principle: **short gaps can be reconstructed, long outages cannot.** PM2.5 is strongly autocorrelated from hour to hour, so a 1-3 hour gap can be filled by linear interpolation with little error (Junninen et al., 2004), while a multi-day outage must stay missing because filling it would invent data.

In [1]:
from pathlib import Path
import sys, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Locate the project root (works whether the kernel starts in notebooks/ or the root).
PROJECT = Path.cwd()
while not (PROJECT / "src" / "aq_common.py").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / "src"))
import aq_common as A

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)

A.set_stage("prep")
A.banner("STEP 1 / 6  -  Data preparation: load, audit, clean, hourly grid")
LOG = A.PipelineLog("step1_dataprep")
print(pd.DataFrame(A.env_facts()["packages"].items(), columns=["package", "version"])
      .to_string(index=False))


  STEP 1 / 6  -  Data preparation: load, audit, clean, hourly grid


2026-10-18 12:37:43 | INFO | === step1_dataprep ===
     package version
       numpy   2.4.4
      pandas   3.0.2
scikit-learn   1.8.0
  matplotlib  3.10.8


## 1. Load and audit the raw file

We read the file exactly as delivered and look at it before changing anything. The audit is deliberately explicit about *zeros*: in this dataset a raw `0` in temperature, humidity, PM or the coordinates is a sentinel for 'no reading', not a real measurement. Detecting that is the first cleaning decision.

In [2]:
banner = "=" * 78
print(banner)
print(" 1. LOAD AND AUDIT THE RAW FILE")
print(banner)

RAW = A.canonicalise(A.load_raw())
LOG.log(f"Raw shape: {RAW.shape}")
print(f"file        : {A.resolve_raw_csv()}")
print(f"shape       : {RAW.shape[0]:,} rows x {RAW.shape[1]} columns")
print(f"memory      : {RAW.memory_usage(deep=True).sum()/1e6:.1f} MB")

print("\nColumns and types")
audit_dtypes = pd.DataFrame({"dtype": RAW.dtypes.astype(str),
                             "example": [RAW[c].dropna().iloc[0] if RAW[c].notna().any() else None
                                         for c in RAW.columns]})
print(audit_dtypes.to_string())

print("\nMissing values and raw zeros (a zero means 'no reading', brief section 5)")
nulls = pd.DataFrame({
    "missing": RAW.isna().sum(),
    "missing_pct": (100 * RAW.isna().mean()).round(2),
})
zeros = {}
for c in A.ZERO_AS_NULL:
    if c in RAW.columns:
        zeros[c] = int((pd.to_numeric(RAW[c], errors="coerce") == 0).sum())
nulls["raw_zeros"] = pd.Series(zeros)
print(nulls.to_string())

print("\nDistinct identifiers")
print(f"  sensors (device_name) : {RAW['device_name'].nunique():,}")
print(f"  sites   (site_name)   : {RAW['site_name'].nunique():,}")
print(f"  networks              : {sorted(RAW['network'].dropna().unique().tolist())}")
print(f"  frequency             : {sorted(RAW['frequency'].dropna().unique().tolist())}")

dups = int(RAW.duplicated(["device_name", "datetime"]).sum())
print(f"\nDuplicate (device, timestamp) rows : {dups}")

nulls.to_csv(A.stage_dir("prep") / "null_report.csv")

 1. LOAD AND AUDIT THE RAW FILE
2026-10-18 12:37:43 | INFO | Raw shape: (14600, 12)
file        : C:\Users\hp\OneDrive\Desktop\3rd YEAR 1st SEM\AI Model Development\Assignment 4\Sensor Data Jan - July 2026.csv
shape       : 14,600 rows x 12 columns
memory      : 2.2 MB

Columns and types
                      dtype                   example
site_name            string               Salama Road
datetime     datetime64[us]       2026-01-01 03:00:00
frequency            string                    hourly
network              string               airgradient
latitude            float64                   0.33354
longitude           float64                  32.56861
temp                float64                    20.535
humidity            float64                    88.405
pm25                float64                    181.16
pm10                float64                    197.89
site_id              string  694140834973ef0013d1a9b0
device_name          string           ag_588c81267534

Missing 


Duplicate (device, timestamp) rows : 0


## 2. When was the data actually collected?

This is the single most important finding in the notebook. The file is named *January - July 2026*, and a reader would assume a continuous seven-month series. It is not. The readings exist for only a few short windows, which changes what kind of model is honest to build and how validation must be done.

In [3]:
print(banner)
print(" 2. WHEN WAS THE DATA COLLECTED?  (the surprise inside the file)")
print(banner)

print(f"UTC source  : the raw 'datetime' column carries a trailing 'Z' (UTC)")
print(f"Kampala span: {RAW['datetime'].min()} .. {RAW['datetime'].max()}")
print(f"distinct hours        : {RAW['datetime'].nunique():,}")
print(f"distinct calendar days: {RAW['datetime'].dt.date.nunique()}")

per_day = (RAW.groupby(RAW["datetime"].dt.date)
              .size().rename("rows").to_frame())
per_day["sensors"] = RAW.groupby(RAW["datetime"].dt.date)["device_name"].nunique()
print("\nReadings per calendar day:")
print(per_day.to_string())
LOG.log(f"Data spans only {per_day.shape[0]} calendar days across "
        f"{RAW['datetime'].dt.to_period('M').nunique()} months")
print("\nINTERPRETATION: although the file is named 'January - July 2026', readings")
print("exist for only three short windows (early Jan, early Apr, early Jun). There is")
print("no July data. Any model is therefore a spatial/temporal *snapshot* model, and")
print("the validation split must respect the window structure (see Step 3).")
per_day.to_csv(A.stage_dir("prep") / "month_coverage.csv")

 2. WHEN WAS THE DATA COLLECTED?  (the surprise inside the file)
UTC source  : the raw 'datetime' column carries a trailing 'Z' (UTC)
Kampala span: 2026-01-01 03:00:00 .. 2026-06-03 06:00:00
distinct hours        : 147
distinct calendar days: 9

Readings per calendar day:
            rows  sensors
datetime                 
2026-01-01  2091      128
2026-01-02  2566      124
2026-01-03   343      111
2026-04-01  2091      122
2026-04-02  2446      123
2026-04-03    63       63
2026-06-01  2067      118
2026-06-02  2374      119
2026-06-03   559       98
2026-10-18 12:37:43 | INFO | Data spans only 9 calendar days across 3 months

INTERPRETATION: although the file is named 'January - July 2026', readings
exist for only three short windows (early Jan, early Apr, early Jun). There is
no July data. Any model is therefore a spatial/temporal *snapshot* model, and
the validation split must respect the window structure (see Step 3).


## 3. Cleaning rules

Five rules, each traceable to the brief or the air-quality literature:

1. **Zeros to null** - a sentinel is not a measurement (brief section 5).
2. **Coordinate repair** - a fixed monitor does not move, so the median position is robust; impossible GPS fixes are dropped.
3. **Physical limits** - above the instrument's range the number is a fault.
4. **Statistical outliers** - flagged but **kept**, because the spikes are real pollution events. Removing them is the dangerous direction for health alerts.
5. **Drop uninformative columns** - `site_id` is half empty; `frequency` is constant. Types are normalised (Kampala time, floats, clean strings).

In [4]:
print(banner)
print(" 3. CLEANING - every rule and why")
print(banner)

from collections import OrderedDict
qc = OrderedDict()                       # rows changed / dropped at each rule
clean = RAW.copy()
qc["raw rows"] = len(clean)

# 3a. zeros -> null -------------------------------------------------------------
before = clean[A.ZERO_AS_NULL[2]].notna().sum() if A.ZERO_AS_NULL[2] in clean else 0
clean = A.zero_to_null(clean)
qc["zeros set to null (temp/RH/PM/coords)"] = int(
    sum((pd.to_numeric(RAW[c], errors="coerce") == 0).sum()
        for c in A.ZERO_AS_NULL if c in RAW.columns))
print(f"[3a] Zeros turned into nulls in {A.ZERO_AS_NULL}.")
print("     A raw 0 in temperature or humidity is impossible in Uganda; a 0 in")
print("     latitude/longitude is the Atlantic, not Kampala. Brief section 5.")
print(f"     -> {qc['zeros set to null (temp/RH/PM/coords)']:,} cell values nulled.")

 3. CLEANING - every rule and why
[3a] Zeros turned into nulls in ['temp', 'humidity', 'pm25', 'pm10', 'latitude', 'longitude'].
     A raw 0 in temperature or humidity is impossible in Uganda; a 0 in
     latitude/longitude is the Atlantic, not Kampala. Brief section 5.
     -> 14,380 cell values nulled.


In [5]:
# 3b. coordinates ---------------------------------------------------------------
clean, coord_ledger = A.fix_coordinates(clean)
print("\n[3b] Coordinate repair:")
print(coord_ledger.to_string(index=False))
print("     A fixed monitor does not move, so each device's median longitude/latitude")
print("     is its best position estimate. GPS fixes > 0.05 deg (~5.5 km) from that")
print("     median are corrupt and are discarded. Devices that never reported a usable")
print("     fix borrow the median position of their site.")
coord_ledger.to_csv(A.stage_dir("prep") / "coordinate_ledger.csv", index=False)


[3b] Coordinate repair:
                                      rule  rows
     GPS fix > 0.05 deg from device median   103
device with no fix -> site median position  2924
     A fixed monitor does not move, so each device's median longitude/latitude
     is its best position estimate. GPS fixes > 0.05 deg (~5.5 km) from that
     median are corrupt and are discarded. Devices that never reported a usable
     fix borrow the median position of their site.


In [6]:
# 3c. physical limits -----------------------------------------------------------
clean, limits_tbl = A.apply_physical_limits(clean)
print("\n[3c] Physical-range enforcement (outside range -> null, not pollution):")
print(limits_tbl.to_string(index=False))
print("     Above the instrument's range the number is an error, not a reading. The")
print("     literature (Okure et al. 2022; Adong et al. 2022) removes such values.")
limits_tbl.to_csv(A.stage_dir("prep") / "physical_limits.csv", index=False)


[3c] Physical-range enforcement (outside range -> null, not pollution):
  column      range  rows_out_of_range
    temp   5.0-45.0                  0
humidity  1.0-100.0                  0
    pm25  0.0-500.0                  3
    pm10 0.0-1000.0                  0
     Above the instrument's range the number is an error, not a reading. The
     literature (Okure et al. 2022; Adong et al. 2022) removes such values.


In [7]:
# 3d. statistical outliers: flag but KEEP --------------------------------------
clean, outlier_tbl = A.flag_robust_outliers(clean, "pm25", "device_name")
top = outlier_tbl.sort_values("outliers", ascending=False).head(10)
print("\n[3d] Per-sensor robust-Z outliers (|z| > 3.5): flagged, NOT removed.")
print(top.to_string(index=False))
print(f"\n     Total flagged hours: {int(outlier_tbl['outliers'].sum()):,} "
      f"({100*outlier_tbl['outliers'].sum()/outlier_tbl['hours'].sum():.2f}% of observed).")
print("     Spikes cluster at the 7-10 am and 7-11 pm peaks (traffic + charcoal")
print("     cooking), so they are real pollution events. Deleting them would teach the")
print("     model that Kampala is cleaner than it is - the dangerous direction for an")
print("     early-warning system. We keep them and flag them for the report.")
outlier_tbl.sort_values("outliers", ascending=False).to_csv(
    A.stage_dir("prep") / "outlier_summary.csv", index=False)


[3d] Per-sensor robust-Z outliers (|z| > 3.5): flagged, NOT removed.
device_name  hours  outliers  max_pm25  outlier_pct
    aq_g5_5    141        39     99.34        27.66
    aq_g519    147        37    105.93        25.17
  aq_g5_101    135        35    116.79        25.93
   aq_g5_42    129        35    118.50        27.13
    aq_g517    145        34     84.15        23.45
    aq_g535    145        33    161.74        22.76
airqo_g5477    135        31    111.47        22.96
  aq_g5_117    145        27    158.65        18.62
    aq_g515    144        25    147.25        17.36
airqo_g5569    139        25    111.42        17.99

     Total flagged hours: 1,161 (7.95% of observed).
     Spikes cluster at the 7-10 am and 7-11 pm peaks (traffic + charcoal
     cooking), so they are real pollution events. Deleting them would teach the
     model that Kampala is cleaner than it is - the dangerous direction for an
     early-warning system. We keep them and flag them for the report.


In [8]:
# 3e. drop uninformative columns, fix types ------------------------------------
drop_present = [c for c in A.DROP_COLUMNS if c in clean.columns]
clean = clean.drop(columns=drop_present)
clean["network"] = clean["network"].astype("string").str.strip().str.lower()
clean["device_name"] = clean["device_name"].astype("string").str.strip()
clean["site_name"] = clean["site_name"].astype("string").str.strip()
for c in ("temp", "humidity", "pm25", "pm10", "latitude", "longitude"):
    clean[c] = pd.to_numeric(clean[c], errors="coerce")
print(f"\n[3e] Dropped uninformative columns: {drop_present}")
print("     site_id  -> 51% empty and duplicates device_name")
print("     frequency -> constant 'hourly', zero information")
print(f"     Rows still present: {len(clean):,}")

# Observed table = rows that still carry a real PM2.5 reading.
observed = clean[clean["pm25"].notna()].copy()
qc["observed rows (PM2.5 present)"] = len(observed)
qc["rows flagged as robust outliers"] = int(observed["flag_outlier"].sum())
print(f"\nObserved rows with a usable PM2.5 value: {len(observed):,}")
LOG.log(f"Clean observed rows: {len(observed)} | PM2.5 non-null: {len(observed)}")
clean.to_parquet(A.stage_dir("prep") / "clean_observed.parquet", index=False)
observed.to_csv(A.stage_dir("prep") / "clean_observed.csv", index=False)


[3e] Dropped uninformative columns: ['site_id', 'frequency']
     site_id  -> 51% empty and duplicates device_name
     frequency -> constant 'hourly', zero information
     Rows still present: 14,600

Observed rows with a usable PM2.5 value: 14,272
2026-10-18 12:37:44 | INFO | Clean observed rows: 14272 | PM2.5 non-null: 14272


## 4. The hourly grid and short-gap interpolation

The brief requires offline hours to be **missing, not absent**. We therefore build the full cartesian product of sensor x hour (149 x 5,088) and left-join the observations onto it. `is_observed` records which sensor-hours were real. Then we linearly interpolate only inside gaps of three hours or fewer, marking every filled cell with `pm25_imputed` so nothing is passed off as measured.

In [9]:
print(banner)
print(" 4. HOURLY GRID  +  SHORT-GAP INTERPOLATION")
print(banner)

grid = A.build_hourly_grid(observed)
expected = observed["device_name"].nunique() * len(
    pd.date_range(A.GRID_START, A.GRID_END, freq="h"))
print(f"Grid: {observed['device_name'].nunique()} sensors x "
      f"{len(pd.date_range(A.GRID_START, A.GRID_END, freq='h')):,} hours "
      f"= {len(grid):,} rows (expected {expected:,}).")
print("The brief requires offline hours to be MISSING, not absent, so every sensor")
print("has a row for every hour; is_observed marks the hours with a real reading.")

grid, gap_ledger = A.fill_short_gaps(grid)
print("\nShort-gap interpolation (linear, only gaps <= 3 hours):")
print(gap_ledger.to_string(index=False))
print("PM2.5 is strongly autocorrelated hour to hour, so a 1-3 hour gap is")
print("reconstructible (Junninen et al., 2004). Longer outages stay missing because")
print("inventing them would be dishonest. Interpolated cells are marked pm25_imputed.")
gap_ledger.to_csv(A.stage_dir("prep") / "gap_summary.csv", index=False)

obs_share = 100 * grid["is_observed"].mean()
print(f"\nObserved sensor-hours: {grid['is_observed'].sum():,} "
      f"({obs_share:.2f}% of the grid); imputed: {int(grid['pm25_imputed'].sum()):,}.")
LOG.log(f"Hourly grid: {len(grid):,} rows | observed {obs_share:.2f}%")

 4. HOURLY GRID  +  SHORT-GAP INTERPOLATION


Grid: 149 sensors x 5,088 hours = 758,112 rows (expected 758,112).
The brief requires offline hours to be MISSING, not absent, so every sensor
has a row for every hour; is_observed marks the hours with a real reading.



Short-gap interpolation (linear, only gaps <= 3 hours):
column  hours_filled
  pm25          1976
  pm10          2029
PM2.5 is strongly autocorrelated hour to hour, so a 1-3 hour gap is
reconstructible (Junninen et al., 2004). Longer outages stay missing because
inventing them would be dishonest. Interpolated cells are marked pm25_imputed.

Observed sensor-hours: 14,272 (1.88% of the grid); imputed: 1,976.
2026-10-18 12:37:46 | INFO | Hourly grid: 758,112 rows | observed 1.88%


## 5. Save the prepared tables and the data dictionary

Everything downstream reads `hourly_clean.parquet`. The QC ledger records how many rows/values each rule touched, so the report can quote exact numbers.

In [10]:
print(banner)
print(" 5. SAVE THE PREPARED TABLES AND WRITE THE DATA DICTIONARY")
print(banner)

grid["hour"] = grid["datetime"].dt.hour
grid["dayofweek"] = grid["datetime"].dt.dayofweek
grid["month"] = grid["datetime"].dt.month

keep = ["device_name", "datetime", "site_name", "network", "latitude", "longitude",
        "temp", "humidity", "pm25", "pm10", "pm25_filled", "pm10_filled",
        "is_observed", "pm25_imputed", "flag_outlier", "hour", "dayofweek", "month"]
grid[keep].to_parquet(A.stage_dir("prep") / "hourly_clean.parquet", index=False)
print(f"wrote hourly_clean.parquet: {grid[keep].shape}")

# 5b. Drop null readings (complete-case modelling table) --------------------------
# The lecturer's feedback: "drop sensors with null values". We checked first: no
# device is empty for the whole period - every one of the 149 sensors has readings -
# so the meaningful form of the rule is to drop the *readings* whose predictors are
# null, i.e. do NOT impute weather. The model then trains only on complete rows.
sensors_total = int(grid["device_name"].nunique())
sensors_no_readings = int((grid.groupby("device_name")["is_observed"].sum() == 0).sum())
qc["sensors without any readings"] = sensors_no_readings
print(f"\n[5b] Sensors with no readings at all: {sensors_no_readings} of {sensors_total} "
      f"(nothing to drop).")

mod = grid[grid["pm25_filled"].notna()].copy()
n_before = len(mod)
mod = mod.dropna(subset=["temp", "humidity", "latitude", "longitude"]).reset_index(drop=True)
n_dropped = n_before - len(mod)
qc["modelling rows (complete-case)"] = int(len(mod))
qc["null readings dropped (temp/humidity/coords)"] = int(n_dropped)
print(f"     Modelling table: kept {len(mod):,} complete rows, dropped {n_dropped:,} "
      f"null readings; sensors kept {mod['device_name'].nunique()} "
      f"(of {sensors_total}).")
print("     Rows with a null temperature, humidity or coordinate are dropped, so no")
print("     imputation is used for the model. This removes the 6 sensors that never")
print("     reported a GPS fix - exactly the 'drop sensors with null values' rule.")
mod[keep].to_parquet(A.stage_dir("prep") / "modelling_ready.parquet", index=False)
print(f"     wrote modelling_ready.parquet: {mod[keep].shape} (nulls left: {int(mod[keep].isna().sum().sum())})")

DATA_DICTIONARY = pd.DataFrame([
    ("device_name", "text", "unique sensor id; the join key across all stages"),
    ("datetime", "datetime (Kampala, UTC+3)", "hourly timestamp, local wall-clock"),
    ("site_name", "text", "site the sensor is installed at"),
    ("network", "text", "hardware network: airqo or airgradient"),
    ("latitude", "float", "repaired sensor latitude (median of valid fixes)"),
    ("longitude", "float", "repaired sensor longitude"),
    ("temp", "float", "air temperature, degC (zeros/out-of-range -> null)"),
    ("humidity", "float", "relative humidity, % (zeros -> null)"),
    ("pm25", "float", "calibrated PM2.5, ug/m3 (observed only; null if offline)"),
    ("pm10", "float", "calibrated PM10, ug/m3 (observed only)"),
    ("pm25_filled", "float", "pm25 with <=3 h gaps linearly interpolated"),
    ("pm10_filled", "float", "pm10 with <=3 h gaps linearly interpolated"),
    ("is_observed", "bool", "True where a real PM2.5 reading exists"),
    ("pm25_imputed", "bool", "True where pm25_filled was interpolated, not observed"),
    ("flag_outlier", "bool", "per-sensor robust-Z |z|>3.5 (kept as a real event)"),
    ("hour", "int", "hour of day (0-23), Kampala time"),
    ("dayofweek", "int", "0=Monday .. 6=Sunday"),
    ("month", "int", "calendar month (1-12)"),
], columns=["column", "type", "description"])
DATA_DICTIONARY.to_csv(A.stage_dir("prep") / "data_dictionary.csv", index=False)
(A.stage_dir("prep") / "data_dictionary.md").write_text(
    "# Data dictionary - cleaned hourly table\n\n" + A.df_to_markdown(DATA_DICTIONARY),
    encoding="utf-8")
print("\nData dictionary")
print(DATA_DICTIONARY.to_string(index=False))

qc_df = pd.DataFrame({"stage": list(qc.keys()), "value": list(qc.values())})
qc_df.to_csv(A.stage_dir("prep") / "qc_counts.csv", index=False)

prep_meta = {
    "raw_file": str(A.resolve_raw_csv()),
    "raw_shape": list(RAW.shape),
    "raw_span_local": [str(RAW["datetime"].min()), str(RAW["datetime"].max())],
    "dates_present": [str(d) for d in sorted(RAW["datetime"].dt.date.unique())],
    "n_sensors": int(observed["device_name"].nunique()),
    "n_sites": int(observed["site_name"].nunique()),
    "networks": sorted(observed["network"].dropna().unique().tolist()),
    "n_observed": int(len(observed)),
    "grid_rows": int(len(grid)),
    "modelling_rows": int(len(mod)),
    "modelling_sensors": int(mod["device_name"].nunique()),
    "sensors_no_readings": int(sensors_no_readings),
    "null_readings_dropped": int(n_before - len(mod)),
    "observed_share_pct": round(float(obs_share), 3),
    "pm25_filled_rows": int(grid["pm25_imputed"].sum()),
    "outliers_flagged": int(observed["flag_outlier"].sum()),
    "qc": qc,
    "cleaning_rules": [
        "zeros in temp/RH/PM/coords -> null (brief section 5)",
        "coordinates: per-device median; fixes >0.05 deg rejected; fallback to site median",
        "physical limits: temp 5-45 C, RH 1-100 %, PM2.5 0-500, PM10 0-1000 ug/m3",
        "per-sensor robust-Z |z|>3.5 flagged but kept (real pollution events)",
        "dropped site_id (51% empty) and frequency (constant)",
        "UTC -> Africa/Kampala (UTC+3); hourly grid; <=3 h gaps linearly interpolated",
        "modelling table drops null readings (complete-case); no weather imputation",
    ],
    "env": A.env_facts(),
}
A.save_json(prep_meta, A.stage_dir("prep") / "prep_meta.json")
LOG.log("Wrote prep_meta.json and data dictionary")
print("\nprep_meta.json summary:")
print(json.dumps({k: prep_meta[k] for k in
                  ["n_sensors", "n_sites", "n_observed", "grid_rows",
                   "observed_share_pct", "outliers_flagged"]}, indent=2))

 5. SAVE THE PREPARED TABLES AND WRITE THE DATA DICTIONARY


wrote hourly_clean.parquet: (758112, 18)

[5b] Sensors with no readings at all: 0 of 149 (nothing to drop).
     Modelling table: kept 9,886 complete rows, dropped 6,362 null readings; sensors kept 136 (of 149).
     Rows with a null temperature, humidity or coordinate are dropped, so no
     imputation is used for the model. This removes the 6 sensors that never
     reported a GPS fix - exactly the 'drop sensors with null values' rule.
     wrote modelling_ready.parquet: (9886, 18) (nulls left: 121)

Data dictionary
      column                      type                                              description
 device_name                      text         unique sensor id; the join key across all stages
    datetime datetime (Kampala, UTC+3)                       hourly timestamp, local wall-clock
   site_name                      text                          site the sensor is installed at
     network                      text                   hardware network: airqo or airgradi

## 6. Figures

In [11]:
print(banner)
print(" 6. FIGURES")
print(banner)

# Fig 1.1 - coverage: how much of the year actually has data?
fig, ax = plt.subplots(figsize=(13, 4))
daily = grid.groupby(grid["datetime"].dt.date).agg(
    observed=("is_observed", "sum")).reset_index()
daily.columns = ["date", "observed"]
ax.bar(daily["date"], daily["observed"], color="#4c72b0", width=0.8)
ax.set_ylabel("observed sensor-hours")
ax.set_xlabel("date (2026)")
ax.set_title("Fig 1.1  Data coverage - only a handful of days in Jan, Apr and Jun exist")
ax.grid(axis="y", alpha=0.3)
A.savefig(fig, "fig1_1_coverage.png")

# Fig 1.2 - availability heatmap: sensor x day
pivot = (grid.assign(day=grid["datetime"].dt.date)
             .groupby(["device_name", "day"])["is_observed"].mean().unstack("day"))
order = pivot.sum(axis=1).sort_values(ascending=False).index
pivot = pivot.loc[order]
fig, ax = plt.subplots(figsize=(13, 9))
im = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels(pivot.index, fontsize=4)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns], rotation=90, fontsize=7)
ax.set_title("Fig 1.2  Sensor availability (share of the 24 hours observed, per day)")
ax.set_ylabel("sensor (device_name)")
cbar = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
cbar.set_label("share of day observed")
fig.tight_layout()
A.savefig(fig, "fig1_2_availability_heatmap.png")
print("Saved fig1_1_coverage.png and fig1_2_availability_heatmap.png")

 6. FIGURES


Saved fig1_1_coverage.png and fig1_2_availability_heatmap.png


In [12]:
summary = LOG.save({
    "stage": "data preparation",
    "rows_raw": int(len(RAW)),
    "rows_observed": int(len(observed)),
    "grid_rows": int(len(grid)),
    "sensors": int(observed["device_name"].nunique()),
})
print("\n" + "=" * 78)
print("  STEP 1 COMPLETE")
print("=" * 78)
print("Outputs in outputs/01_prep/ :")
for f in sorted(A.stage_dir("prep").glob("*")):
    print(f"  {f.name:34s} {f.stat().st_size:>12,d} bytes")
print("\nNext:  Step2_EDA.ipynb reads hourly_clean.parquet")


  STEP 1 COMPLETE
Outputs in outputs/01_prep/ :
  clean_observed.csv                    1,893,948 bytes
  clean_observed.parquet                  456,854 bytes
  coordinate_ledger.csv                       103 bytes
  data_dictionary.csv                       1,060 bytes
  data_dictionary.md                        1,253 bytes
  fig1_1_coverage.png                      40,321 bytes
  fig1_2_availability_heatmap.png         237,646 bytes
  gap_summary.csv                              43 bytes
  hourly_clean.parquet                  1,296,978 bytes
  modelling_ready.parquet                 292,456 bytes
  month_coverage.csv                          206 bytes
  null_report.csv                             275 bytes
  outlier_summary.csv                       4,256 bytes
  physical_limits.csv                         108 bytes
  prep_meta.json                            2,053 bytes
  qc_counts.csv                               269 bytes

Next:  Step2_EDA.ipynb reads hourly_clean.parquet
